# 01 EDA

**Owner:** Person A.

**Inputs:** `data/raw/`.

**Outputs:** `outputs/tables/profile_*.csv` and `outputs/figures/missing_*.png`.

**Stop rule:** if `data/raw` is empty, stop. Do not invent columns. Two or three figures are enough for the slides: coverage, a key distribution, and missingness.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import io as sio

files = sio.list_raw_files(RAW)
if not files:
    print("STOP: no files in data/raw.")
else:
    for path in files:
        frame = sio.load_table(path)
        prof = sio.profile(frame)
        saved = sio.write_profile(frame, path.stem)
        print(path.name, "rows", len(frame), "columns", frame.shape[1], "->", saved.name)
        fig, ax = plt.subplots(figsize=(8, max(3, 0.28 * len(prof))))
        order = prof.sort_values("missing_pct")
        ax.barh(order["column"].astype(str), order["missing_pct"])
        ax.set_xlim(0, 1)
        ax.set_xlabel("missing fraction")
        ax.set_title(path.name)
        figure = sio.save_figure(fig, f"missing_{path.stem}.png")
        print(" ", figure.name)
